# Set Configuration

In [ ]:
from diffusion_hash_inv.config import MainConfig, HashConfig, MessageConfig, OutputConfig, Byte2RGBConfig
from diffusion_hash_inv.main import MainEP
from diffusion_hash_inv.utils import FileIO
from diffusion_hash_inv.main import RuntimeConfig

length = 16
iteration = 2**length

main_cfg = MainConfig(
    verbose_flag=False,
    clean_flag=True,
    debug_flag=False,
    make_image_flag=False,
)
hash_cfg = HashConfig(
    hash_alg="md5",
    length=length,
)
message_cfg = MessageConfig(
    message_flag=False,
    length=length,
    random_flag=False,
    seed_flag=True,
)
output_cfg = OutputConfig()
byte2rgb_cfg = Byte2RGBConfig()
rutime_cfg = RuntimeConfig(
    main=main_cfg,
    message=message_cfg,
    hash=hash_cfg,
    output=output_cfg,
    rgb=byte2rgb_cfg,
)

io_controller = FileIO(main_config=main_cfg, output_cfg=output_cfg)

In [ ]:
main = MainEP(runtime_config=rutime_cfg, file_controller=io_controller)
main.run(iteration=iteration, mode="sequential")

In [ ]:

from diffusion_hash_inv.logger.logger import Logs

log_hierarchy = list()
logs = Logs.get_logs(io_controller, hash_cfg, main_cfg, log_hierarchy)
print(len(logs))

In [ ]:
def get_step4(logs):
    step4_logs = []
    for log in logs:
        _tmp = list(log.values())
        assert len(_tmp) == 1, "Each log dictionary should contain exactly one key-value pair."
        log_dict = list(log.values())[0]
        if "Logs" in log_dict and "4th Step" in log_dict["Logs"]:
            step4_logs.append(log_dict["Logs"]["4th Step"])
    return step4_logs

In [ ]:
from typing import List, Dict, Any

step4_logs: List[Dict[str, Any]] = get_step4(logs)
print(step4_logs[0])
beta_schedule = list()

In [ ]:
def cumulative_block(byte_list: List[bytes], block: bytes, indent: int = 0) -> List[bytes]:
    """
    Seperate Block into bytes and cumulatively add to byte_list. Return the updated byte_list.
    """
    _byte = 0
    for byte in block:
        _byte = byte if len(byte_list) == 0 else byte + byte_list[-1]
        _byte = _byte % (0xff + 1) # Ensure byte value stays within 0-255
        if main_cfg.verbose_flag:
            # print(f"{'\t' * indent}Byte: {byte}")
            # print(f"{'\t' * (indent+1)}Cumulative Byte: {_byte}")
            pass
        byte_list.append(_byte)
    return byte_list


def make_beta_schedule(step4_logs: Dict[str, Any]) -> List[float]:
    step4_log: Dict[str, Any] = list(step4_logs.values())
    beta_schedule = []
    for log in step4_log:
        for key, value in log.items():
            if main_cfg.verbose_flag:
                # print(f"Key: {key}")
                pass
            for k, v in value.items():
                _v = Logs.str_to_bytes(v)
                if main_cfg.verbose_flag:
                    # print(f"\tKey: {k}, Value: {v}")
                    # print(f"\t\tConverted Value: {_v}")
                    pass
                cumulative_block(beta_schedule, _v, indent=3)
                
    return beta_schedule

In [ ]:
from tqdm import tqdm

step4_log_process = tqdm(step4_logs, desc="Processing Step 4 Logs")
for log in step4_log_process:
    beta_schedule.append(make_beta_schedule(log))

In [ ]:
print(len(beta_schedule))
print(len(beta_schedule[0]))
print(beta_schedule[0])
print(beta_schedule[1])

In [ ]:
width = len(str(iteration))
for i, beta in enumerate(beta_schedule):
    print(f"Beta Schedule {i:0{width}}:")
    print(f"{'\t' * 1}Beta: ", end="")
    for j, b in enumerate(beta):
        print(f"{b:03},", end=" ")
    print("\n")

In [ ]:
import numpy as np

beta_array = np.array(beta_schedule)
print(beta_array.shape)
min = np.min(beta_array, axis=0)
max = np.max(beta_array, axis=0)
mean = np.mean(beta_array, axis=0)
var = np.var(beta_array, axis=0)
std = np.std(beta_array, axis=0)

np.set_printoptions(threshold=np.inf, linewidth=np.inf)

print(f"Min: {min}, Length: {len(min)}")
print(f"Max: {max}, Length: {len(max)}")
print(f"Mean: {mean}, Length: {len(mean)}")
print(f"Variance: {var}, Length: {len(var)}")
print(f"Standard Deviation: {std}, Length: {len(std)}")